In [2]:
   # !unzip -o -q health_data.zip   # only needed in Colab, after uploading the zip

import pandas as pd
import numpy as np

accounts = pd.read_csv('data/accounts.csv', parse_dates=['contract_start_date'])
tickets  = pd.read_csv('data/tickets.csv',
                       parse_dates=['created_at', 'first_response_at', 'resolved_at'])
truth    = pd.read_csv('truth/account_truth.csv', parse_dates=['churn_date'])

AS_OF = pd.Timestamp('2026-07-31 23:59:59')

# Accounts that cancelled before the as-of date are gone; a CRM would know this
already_churned = truth.loc[truth.churn_date <= AS_OF, 'account_id']
active = accounts[~accounts.account_id.isin(already_churned)].copy()

# Only tickets the business could see on the as-of date
hist = tickets[(tickets.created_at <= AS_OF) &
               tickets.account_id.isin(active.account_id)].copy()

print('Accounts:', accounts.shape, ' Tickets:', tickets.shape)
print('Active at as-of date:', len(active), ' Already churned:', len(already_churned))
print('Tickets visible as of', AS_OF.date(), ':', len(hist))
print('Accounts under 90 days old:',
      (active.contract_start_date > AS_OF - pd.Timedelta(days=90)).sum())

Accounts: (400, 8)  Tickets: (22457, 11)
Active at as-of date: 369  Already churned: 31
Tickets visible as of 2026-07-31 : 16549
Accounts under 90 days old: 10


In [4]:
# --- Signal 1: last-30-day volume vs the account's own baseline ---
recent_start   = AS_OF - pd.Timedelta(days=30)
baseline_start = AS_OF - pd.Timedelta(days=210)    # 6 months before the recent window

sig = active[['account_id', 'account_name', 'segment', 'contract_start_date']].copy().set_index('account_id')

# Tickets in the last 30 days
recent = hist[hist.created_at > recent_start]
sig['tickets_30d'] = recent.groupby('account_id').size()
sig['tickets_30d'] = sig['tickets_30d'].fillna(0)

# Baseline window starts at the later of: 210 days ago, or 60 days after contract start
obs_start = (sig.contract_start_date + pd.Timedelta(days=60)).clip(lower=baseline_start)
baseline_days = (recent_start - obs_start).dt.days.clip(lower=0)

base = hist.merge(obs_start.rename('obs_start'), left_on='account_id', right_index=True)
base = base[(base.created_at > base.obs_start) & (base.created_at <= recent_start)]
sig['baseline_tickets'] = base.groupby('account_id').size()
sig['baseline_tickets'] = sig['baseline_tickets'].fillna(0)

# Baseline expressed as tickets per 30 days; blank if under 60 days of history
sig['baseline_per_30d'] = np.where(baseline_days >= 60,
                                   sig.baseline_tickets / baseline_days * 30, np.nan)

# Ratio (add 1 to both sides so a tiny account going 0 -> 1 ticket isn't a "spike")
sig['volume_ratio'] = (sig.tickets_30d + 1) / (sig.baseline_per_30d + 1)
sig['flag_volume']  = sig.volume_ratio > 1.5

print('Accounts with a usable baseline:', sig.volume_ratio.notna().sum(), 'of', len(sig))
print(sig.volume_ratio.describe().round(2))
print('\nFlagged (> 1.5x own baseline):', sig.flag_volume.sum())
sig.sort_values('volume_ratio', ascending=False)[
    ['account_name', 'segment', 'tickets_30d', 'baseline_per_30d', 'volume_ratio']].head(5).round(2)

Accounts with a usable baseline: 335 of 369
count    335.00
mean       1.07
std        0.52
min        0.20
25%        0.71
50%        1.00
75%        1.36
max        4.00
Name: volume_ratio, dtype: float64

Flagged (> 1.5x own baseline): 55


,account_name,segment,tickets_30d,baseline_per_30d,volume_ratio
account_id,,,,,
ACC-1280,Coastal Cardiology Associates Southwest,Cardiology practice,9.0,1.50,4.00
ACC-1336,Summit Imaging Southeast,Imaging center,7.0,1.67,3.00
ACC-1066,Riverside Diagnostic Imaging,Imaging center,6.0,1.50,2.80
ACC-1316,Clearwater Community Hospital West,Community hospital,6.0,1.67,2.62
ACC-1161,Harbor Medical Group,Hospital system,22.0,7.83,2.60


In [5]:
# --- Signal 2: average satisfaction rating, last 90 days ---
csat_start = AS_OF - pd.Timedelta(days=90)

rated = hist[(hist.resolved_at > csat_start) &
             (hist.resolved_at <= AS_OF) &          # rating must exist by the as-of date
             hist.satisfaction_rating.notna()]

csat = rated.groupby('account_id').satisfaction_rating.agg(['mean', 'count'])
sig['csat_n']   = csat['count']
sig['csat_n']   = sig['csat_n'].fillna(0).astype(int)
sig['csat_avg'] = np.where(sig.csat_n >= 3, csat['mean'].reindex(sig.index), np.nan)
sig['flag_csat'] = sig.csat_avg < 3.0

print('Accounts with 3+ ratings:', sig.csat_avg.notna().sum(), 'of', len(sig))
print(sig.csat_avg.describe().round(2))
print('\nFlagged (avg below 3.0):', sig.flag_csat.sum())
print('Flagged on BOTH volume and CSAT:', (sig.flag_volume & sig.flag_csat).sum())
print()
print(sig[sig.flag_csat].sort_values('csat_avg')[
    ['account_name', 'csat_n', 'csat_avg', 'volume_ratio']].head(5).round(2).to_string())

Accounts with 3+ ratings: 251 of 369
count    251.00
mean       3.91
std        0.54
min        1.86
25%        3.67
50%        4.00
75%        4.29
max        5.00
Name: csat_avg, dtype: float64

Flagged (avg below 3.0): 16
Flagged on BOTH volume and CSAT: 3

                                 account_name  csat_n  csat_avg  volume_ratio
account_id                                                                   
ACC-1378    Hillcrest Regional Medical Center       7      1.86          0.52
ACC-1055                     Highland Imaging       4      2.00          1.14
ACC-1203             Valley Memorial Hospital      10      2.30          1.70
ACC-1148             Summit Memorial Hospital      15      2.33          1.95
ACC-1229           Granite Diagnostic Imaging       7      2.43          1.40


In [6]:
# --- Signal 3: open High/Critical tickets older than 7 days on the as-of date ---
open_asof = hist[hist.resolved_at.isna() | (hist.resolved_at > AS_OF)]   # still open on July 31
age_days  = (AS_OF - open_asof.created_at).dt.days

stale_hipri = open_asof[open_asof.priority.isin(['High', 'Critical']) & (age_days > 7)]
sig['open_hipri_7d'] = stale_hipri.groupby('account_id').size()
sig['open_hipri_7d'] = sig['open_hipri_7d'].fillna(0).astype(int)
sig['flag_open_hipri'] = sig.open_hipri_7d >= 1

print('Tickets open on the as-of date:', len(open_asof))
print('  of those, High/Critical and older than 7 days:', len(stale_hipri))
print('\nAccounts flagged (1+ stale high-priority ticket):', sig.flag_open_hipri.sum())
print(sig.open_hipri_7d.value_counts().sort_index().to_string())
print()
print(sig[sig.flag_open_hipri].sort_values('open_hipri_7d', ascending=False)[
    ['account_name', 'open_hipri_7d', 'csat_avg', 'volume_ratio']].head(5).round(2).to_string())

Tickets open on the as-of date: 216
  of those, High/Critical and older than 7 days: 6

Accounts flagged (1+ stale high-priority ticket): 6
open_hipri_7d
0    363
1      6

                                      account_name  open_hipri_7d  csat_avg  volume_ratio
account_id                                                                               
ACC-1148                  Summit Memorial Hospital              1      2.33          1.95
ACC-1203                  Valley Memorial Hospital              1      2.30          1.70
ACC-1289    Lakeshore Community Hospital Southwest              1      3.60          1.21
ACC-1337                 Harbor Medical Group West              1      2.67          1.50
ACC-1350           Prairie Health System Southeast              1      3.27          1.35


In [7]:
# --- Signal 4: repeat contacts (same category within 7 days), as a share of recent tickets ---
h = hist.sort_values('created_at').copy()
h['prev_same_issue'] = h.groupby(['account_id', 'category']).created_at.shift(1)
h['is_repeat'] = (h.created_at - h.prev_same_issue) <= pd.Timedelta(days=7)

r30 = h[h.created_at > recent_start]
rep = r30.groupby('account_id').is_repeat.agg(['sum', 'count'])
sig['repeats_30d'] = rep['sum']
sig['repeats_30d'] = sig['repeats_30d'].fillna(0).astype(int)
sig['repeat_rate'] = np.where(sig.tickets_30d >= 3, sig.repeats_30d / sig.tickets_30d, np.nan)
sig['flag_repeat'] = (sig.repeat_rate >= 1/3) & (sig.repeats_30d >= 2)

print('Share of all tickets that are 7-day repeats:', round(h.is_repeat.mean(), 3))
print('Accounts with 3+ tickets in last 30 days:', sig.repeat_rate.notna().sum(), 'of', len(sig))
print(sig.repeat_rate.describe().round(2))
print('\nAccounts flagged (1 in 3+ tickets is a repeat, and 2+ repeats):', sig.flag_repeat.sum())
print(sig.groupby('segment').flag_repeat.mean().round(2).to_string())

# --- Signal 5: resolution time, adjusted for priority ---
closed = hist[hist.resolved_at <= AS_OF].copy()
closed['res_hours'] = (closed.resolved_at - closed.created_at).dt.total_seconds() / 3600

# Compare each ticket to the typical time for ITS priority (Critical vs Critical, Low vs Low)
pri_median = closed.groupby('priority').res_hours.median()
closed['res_vs_typical'] = closed.res_hours / closed.priority.map(pri_median)

recent_closed = closed[closed.resolved_at > csat_start]          # last 90 days
res = recent_closed.groupby('account_id').res_vs_typical.agg(['median', 'count'])
sig['res_n'] = res['count']
sig['res_n'] = sig['res_n'].fillna(0).astype(int)
sig['res_ratio'] = np.where(sig.res_n >= 3, res['median'].reindex(sig.index), np.nan)

res_p75 = sig.res_ratio.quantile(0.75)
sig['flag_slow'] = sig.res_ratio > res_p75

print('\nTypical (median) resolution hours by priority:')
print(pri_median.reindex(['Critical', 'High', 'Medium', 'Low']).round(1).to_string())
print('\nAccounts with 3+ resolved tickets:', sig.res_ratio.notna().sum(), 'of', len(sig))
print('75th percentile of res_ratio:', round(res_p75, 2))
print('Accounts flagged (slower than 75th pct):', sig.flag_slow.sum())

# --- How many signals does each account trip? ---
flags = ['flag_volume', 'flag_csat', 'flag_open_hipri', 'flag_repeat', 'flag_slow']
sig['n_flags'] = sig[flags].sum(axis=1)
print('\nNumber of flags per account:')
print(sig.n_flags.value_counts().sort_index().to_string())

Share of all tickets that are 7-day repeats: 0.164
Accounts with 3+ tickets in last 30 days: 220 of 369
count    220.00
mean       0.16
std        0.16
min        0.00
25%        0.00
50%        0.14
75%        0.27
max        0.62
Name: repeat_rate, dtype: float64

Accounts flagged (1 in 3+ tickets is a repeat, and 2+ repeats): 31
segment
Cardiology practice    0.03
Community hospital     0.08
Hospital system        0.15
Imaging center         0.05

Typical (median) resolution hours by priority:
priority
Critical     6.4
High        16.7
Medium      37.8
Low         72.2

Accounts with 3+ resolved tickets: 353 of 369
75th percentile of res_ratio: 1.38
Accounts flagged (slower than 75th pct): 88

Number of flags per account:
n_flags
0    221
1    110
2     31
3      5
4      1
5      1


In [8]:
# --- Step 6: combine signals into a 0-100 health score ---

# 1. Turn each signal into a 0-1 penalty (0 = fine, 1 = as bad as it gets)
penalty = pd.DataFrame(index=sig.index)
penalty['Ticket volume spike']        = ((sig.volume_ratio - 1.0) / 1.0).clip(0, 1)   # 1x -> 0, 2x+ -> 1
penalty['Low satisfaction']           = ((4.0 - sig.csat_avg) / 2.0).clip(0, 1)       # 4.0 -> 0, 2.0 -> 1
penalty['Stale high-priority ticket'] = (sig.open_hipri_7d >= 1).astype(float)
penalty['Repeat contacts']            = ((sig.repeat_rate - 0.10) / 0.40).clip(0, 1)  # 10% -> 0, 50% -> 1
penalty['Slow resolution']            = ((sig.res_ratio - 1.0) / 1.0).clip(0, 1)      # typical -> 0, 2x -> 1
penalty = penalty.fillna(0)              # blank signal = no evidence = no penalty

# 2. Weights (must sum to 100)
WEIGHTS = {'Ticket volume spike': 20, 'Low satisfaction': 30,
           'Stale high-priority ticket': 15, 'Repeat contacts': 15, 'Slow resolution': 20}
points_lost = penalty * pd.Series(WEIGHTS)

# 3. Score, band, and the single biggest reason for each account
sig['health_score'] = (100 - points_lost.sum(axis=1)).round(1)
sig['band'] = pd.cut(sig.health_score, bins=[0, 60, 80, 100.1], right=False,
                     labels=['Red', 'Yellow', 'Green'])
sig['top_signal'] = points_lost.idxmax(axis=1).where(points_lost.max(axis=1) > 0, 'None')
sig['top_signal_points'] = points_lost.max(axis=1).round(1)

print('Weights (sum = %d):' % sum(WEIGHTS.values()), WEIGHTS)
print('\nHealth score distribution:')
print(sig.health_score.describe().round(1).to_string())
print('\nBand counts:')
print(sig.band.value_counts().reindex(['Green', 'Yellow', 'Red']).to_string())
print('\nRed accounts, worst first:')
print(sig[sig.band == 'Red'].sort_values('health_score')[
    ['account_name', 'segment', 'health_score', 'top_signal', 'top_signal_points']].to_string())

Weights (sum = 100): {'Ticket volume spike': 20, 'Low satisfaction': 30, 'Stale high-priority ticket': 15, 'Repeat contacts': 15, 'Slow resolution': 20}

Health score distribution:
count    369.0
mean      86.7
std       13.8
min       10.6
25%       80.0
50%       90.0
75%       98.6
max      100.0

Band counts:
band
Green     277
Yellow     75
Red        17

Red accounts, worst first:
                                        account_name              segment  health_score                  top_signal  top_signal_points
account_id                                                                                                                            
ACC-1203                    Valley Memorial Hospital   Community hospital          10.6            Low satisfaction               25.5
ACC-1148                    Summit Memorial Hospital   Community hospital          28.8            Low satisfaction               25.0
ACC-1160        Lakeshore Radiology Partners Midwest       Imaging cen

In [9]:
# --- Step 7: evaluate against the hidden truth (the ONLY place truth labels are used) ---
from sklearn.metrics import roc_auc_score

ev = sig.join(truth.set_index('account_id')[['true_risk_state', 'risk_pattern', 'churned', 'churn_date']])
ev['churned_next_90d'] = ev.churned & (ev.churn_date > AS_OF)
ev['at_risk_truth'] = ev.true_risk_state != 'healthy'
risk = 100 - ev.health_score        # higher = riskier, for AUC

print('1) Band vs true risk state (rows = band, cols = truth):')
print(pd.crosstab(ev.band, ev.true_risk_state).reindex(['Green','Yellow','Red'])[['healthy','slipping','at_risk']].to_string())

print('\n2) Churn in the 90 days after the as-of date, by band:')
c = ev.groupby('band', observed=True).churned_next_90d.agg(['sum','count','mean']).reindex(['Green','Yellow','Red'])
c.columns = ['churned','accounts','churn_rate']
c['churn_rate'] = (c.churn_rate*100).round(1).astype(str) + '%'
print(c.to_string())

print('\n3) AUC (0.5 = coin flip, 1.0 = perfect):')
print('   Score vs truly deteriorating :', round(roc_auc_score(ev.at_risk_truth, risk), 2))
print('   Score vs churned in next 90d :', round(roc_auc_score(ev.churned_next_90d, risk), 2))

print('\n4) Deteriorating accounts caught (Red or Yellow), by hidden pattern:')
d = ev[ev.at_risk_truth]
print(d.groupby('risk_pattern').band.apply(lambda b: f"{b.isin(['Red','Yellow']).sum()} of {len(b)}").to_string())

1) Band vs true risk state (rows = band, cols = truth):
true_risk_state  healthy  slipping  at_risk
band                                       
Green                227        31       19
Yellow                40        26        9
Red                    2         6        9

2) Churn in the 90 days after the as-of date, by band:
        churned  accounts churn_rate
band                                
Green        15       277       5.4%
Yellow        5        75       6.7%
Red           1        17       5.9%

3) AUC (0.5 = coin flip, 1.0 = perfect):
   Score vs truly deteriorating : 0.72
   Score vs churned in next 90d : 0.49

4) Deteriorating accounts caught (Red or Yellow), by hidden pattern:
risk_pattern
at_risk     15 of 28
silent        3 of 9
slipping    32 of 63


In [10]:
# --- Step 8a: scoring + evaluation as reusable functions ---
from sklearn.metrics import roc_auc_score

WEIGHTS_V1 = {'Ticket volume spike': 20, 'Low satisfaction': 30, 'Stale high-priority ticket': 15,
              'Repeat contacts': 15, 'Slow resolution': 20}
# v2 = rejected experiment, kept so its results are reproducible
WEIGHTS_V2 = {'Ticket volume spike': 15, 'Low satisfaction': 25, 'Stale high-priority ticket': 10,
              'Repeat contacts': 15, 'Slow resolution': 15, 'Engagement drop': 20}


def score_accounts(as_of, version='v1'):
    """Score every active account using only data visible on the as-of date."""
    AS_OF = pd.Timestamp(as_of) + pd.Timedelta(hours=23, minutes=59, seconds=59)
    gone = truth.loc[truth.churn_date <= AS_OF, 'account_id']           # known cancellations
    active = accounts[~accounts.account_id.isin(gone) &
                      (accounts.contract_start_date <= AS_OF)]           # exclude not-yet-customers
    hist = tickets[(tickets.created_at <= AS_OF) & tickets.account_id.isin(active.account_id)]

    recent_start   = AS_OF - pd.Timedelta(days=30)
    win90_start    = AS_OF - pd.Timedelta(days=90)
    baseline_start = AS_OF - pd.Timedelta(days=210)
    sig = active[['account_id', 'account_name', 'segment', 'contract_start_date']].set_index('account_id')

    # Signal 1: volume spike (last 30d vs own baseline, onboarding excluded)
    sig['tickets_30d'] = hist[hist.created_at > recent_start].groupby('account_id').size()
    sig['tickets_30d'] = sig['tickets_30d'].fillna(0)
    obs_start = (sig.contract_start_date + pd.Timedelta(days=60)).clip(lower=baseline_start)
    baseline_days = (recent_start - obs_start).dt.days.clip(lower=0)
    base = hist.merge(obs_start.rename('obs_start'), left_on='account_id', right_index=True)
    base = base[(base.created_at > base.obs_start) & (base.created_at <= recent_start)]
    sig['baseline_tickets'] = base.groupby('account_id').size()
    sig['baseline_tickets'] = sig['baseline_tickets'].fillna(0)
    sig['baseline_per_30d'] = np.where(baseline_days >= 60, sig.baseline_tickets / baseline_days * 30, np.nan)
    sig['volume_ratio'] = (sig.tickets_30d + 1) / (sig.baseline_per_30d + 1)

    # Signal 2: satisfaction (last 90d, only ratings that existed by the as-of date)
    rated = hist[(hist.resolved_at > win90_start) & (hist.resolved_at <= AS_OF) & hist.satisfaction_rating.notna()]
    cs = rated.groupby('account_id').satisfaction_rating.agg(['mean', 'count', 'sum'])
    sig['csat_n'] = cs['count']
    sig['csat_n'] = sig['csat_n'].fillna(0).astype(int)
    sig['csat_avg'] = np.where(sig.csat_n >= 3, cs['mean'].reindex(sig.index), np.nan)

    # Signal 3: open High/Critical tickets older than 7 days
    open_asof = hist[hist.resolved_at.isna() | (hist.resolved_at > AS_OF)]
    stale = open_asof[open_asof.priority.isin(['High', 'Critical']) &
                      ((AS_OF - open_asof.created_at).dt.days > 7)]
    sig['open_hipri_7d'] = stale.groupby('account_id').size()
    sig['open_hipri_7d'] = sig['open_hipri_7d'].fillna(0).astype(int)

    # Signal 4: repeat contacts (same category within 7 days), share of last-30d tickets
    h = hist.sort_values('created_at').copy()
    h['prev'] = h.groupby(['account_id', 'category']).created_at.shift(1)
    h['is_repeat'] = (h.created_at - h.prev) <= pd.Timedelta(days=7)
    sig['repeats_30d'] = h[h.created_at > recent_start].groupby('account_id').is_repeat.sum()
    sig['repeats_30d'] = sig['repeats_30d'].fillna(0).astype(int)
    sig['repeat_rate'] = np.where(sig.tickets_30d >= 3, sig.repeats_30d / sig.tickets_30d, np.nan)

    # Signal 5: resolution time vs typical for the same priority
    closed = hist[hist.resolved_at <= AS_OF].copy()
    closed['res_hours'] = (closed.resolved_at - closed.created_at).dt.total_seconds() / 3600
    closed['res_vs_typical'] = closed.res_hours / closed.priority.map(closed.groupby('priority').res_hours.median())
    rs = closed[closed.resolved_at > win90_start].groupby('account_id').res_vs_typical.agg(['median', 'count'])
    sig['res_n'] = rs['count']
    sig['res_n'] = sig['res_n'].fillna(0).astype(int)
    sig['res_ratio'] = np.where(sig.res_n >= 3, rs['median'].reindex(sig.index), np.nan)

    # Penalties (0 = fine, 1 = worst)
    p = pd.DataFrame(index=sig.index)
    p['Ticket volume spike']        = ((sig.volume_ratio - 1.0) / 1.0).clip(0, 1)
    p['Stale high-priority ticket'] = (sig.open_hipri_7d >= 1).astype(float)
    p['Repeat contacts']            = ((sig.repeat_rate - 0.10) / 0.40).clip(0, 1)
    p['Slow resolution']            = ((sig.res_ratio - 1.0) / 1.0).clip(0, 1)

    if version == 'v1':
        p['Low satisfaction'] = ((4.0 - sig.csat_avg) / 2.0).clip(0, 1)
        weights = WEIGHTS_V1
    else:
        # v2-A: shrink CSAT toward the overall mean (3 pseudo-ratings) so 1-2 ratings still count
        prior, K = rated.satisfaction_rating.mean(), 3
        sig['csat_adj'] = (cs['sum'].reindex(sig.index).fillna(0) + K * prior) / (sig.csat_n + K)
        p['Low satisfaction'] = ((4.0 - sig.csat_adj) / 2.0).clip(0, 1)
        # v2-B: engagement drop - last 60 days well below own baseline
        sig['tickets_60d'] = hist[hist.created_at > AS_OF - pd.Timedelta(days=60)].groupby('account_id').size()
        sig['tickets_60d'] = sig['tickets_60d'].fillna(0)
        sig['drop_ratio'] = (sig.tickets_60d / 2 + 1) / (sig.baseline_per_30d + 1)
        p['Engagement drop'] = ((0.8 - sig.drop_ratio) / 0.4).clip(0, 1)
        weights = WEIGHTS_V2

    p = p.fillna(0)[list(weights)]
    lost = p * pd.Series(weights)
    sig['health_score'] = (100 - lost.sum(axis=1)).round(1)
    sig['band'] = pd.cut(sig.health_score, bins=[0, 60, 80, 100.1], right=False,
                         labels=['Red', 'Yellow', 'Green']).astype(str)
    sig['top_signal'] = lost.idxmax(axis=1).where(lost.max(axis=1) > 0, 'None')
    if version != 'v1':
        # v2-C: too little data to judge -> label it instead of calling it Green
        n_90d = hist[hist.created_at > win90_start].groupby('account_id').size().reindex(sig.index).fillna(0)
        thin = (n_90d < 3) & (sig.csat_n == 0)
        sig.loc[thin & (sig.band == 'Green'), 'band'] = 'Insufficient data'
    return sig, AS_OF


def evaluate(sig, AS_OF, horizon_days=90):
    """Compare scores to hidden truth. The only function that reads risk labels or future churn."""
    ev = sig.join(truth.set_index('account_id')[['true_risk_state', 'risk_pattern', 'churn_date']])
    ev['churn_next'] = (ev.churn_date > AS_OF) & (ev.churn_date <= AS_OF + pd.Timedelta(days=horizon_days))
    ev['bad'] = ev.true_risk_state != 'healthy'
    risk = 100 - ev.health_score
    order = ['Green', 'Yellow', 'Red', 'Insufficient data']
    tab = ev.groupby('band').agg(accounts=('bad', 'size'), deteriorating=('bad', 'sum'), churned=('churn_next', 'sum'))
    tab = tab.reindex([o for o in order if o in tab.index])
    tab['pct_deteriorating'] = (tab.deteriorating / tab.accounts * 100).round(0)
    tab['churn_rate_pct'] = (tab.churned / tab.accounts * 100).round(1)
    return dict(auc_det=round(roc_auc_score(ev.bad, risk), 2),
                auc_churn=round(roc_auc_score(ev.churn_next, risk), 2),
                tab=tab,
                caught=ev[ev.bad].groupby('risk_pattern').band.apply(
                    lambda b: f"{b.isin(['Red', 'Yellow']).sum()} of {len(b)}"))


# Sanity check: the function must reproduce Steps 6-7 exactly
s, A = score_accounts('2026-07-31', 'v1')
print(s.band.value_counts().reindex(['Green', 'Yellow', 'Red']).to_string())
r = evaluate(s, A)
print('AUC deteriorating:', r['auc_det'], ' AUC churn:', r['auc_churn'])

band
Green     277
Yellow     75
Red        17
AUC deteriorating: 0.72  AUC churn: 0.49


In [11]:
# --- Step 8b: v1 vs v2 on the design date and an unseen test date ---
rows = []
for as_of, role in [('2026-07-31', 'design date'), ('2026-04-30', 'test date')]:
    for v in ['v1', 'v2']:
        s, A = score_accounts(as_of, v)
        r = evaluate(s, A)
        red = r['tab'].loc['Red'] if 'Red' in r['tab'].index else None
        rows.append({'as_of': f'{as_of} ({role})', 'version': v,
                     'AUC_deteriorating': r['auc_det'], 'AUC_churn_90d': r['auc_churn'],
                     'red_accounts': int(red.accounts) if red is not None else 0,
                     'red_pct_deteriorating': f"{red.pct_deteriorating:.0f}%" if red is not None else '-',
                     'silent_caught': r['caught'].get('silent', '-')})
comparison = pd.DataFrame(rows).set_index(['as_of', 'version'])
print(comparison.to_string())

print('\nv1 on the test date, by band:')
s, A = score_accounts('2026-04-30', 'v1')
print(evaluate(s, A)['tab'].to_string())

                                  AUC_deteriorating  AUC_churn_90d  red_accounts red_pct_deteriorating silent_caught
as_of                    version                                                                                    
2026-07-31 (design date) v1                    0.72           0.49            17                   88%        3 of 9
                         v2                    0.70           0.49             8                  100%        3 of 9
2026-04-30 (test date)   v1                    0.59           0.59             5                   80%       1 of 14
                         v2                    0.57           0.59             1                    0%       0 of 14

v1 on the test date, by band:
        accounts  deteriorating  churned  pct_deteriorating  churn_rate_pct
band                                                                       
Green        312             84       11               27.0             3.5
Yellow        59             21        